# **Assignment 4: some database operations**
- **you will learn:** using joins, aggregation, groupby in Pandas
- **task:**  See section 4.2 below
- **deadline:** 15.11.2026
- [Pandas documentation](https://pandas.pydata.org/docs/index.html)
- 📝 **Reminder:** Sync your GitHub repository with the main course repository, update your project in PyCharm, and after completing the assignment, commit and push your changes back to GitHub.

In [67]:
import numpy as np
import pandas as pd

print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)

NumPy version: 2.5.4
Pandas version: 3.0.6


In [68]:
# -------------------------------
# 1) Build synthetic datasets
# -------------------------------
# sensor readouts (long-format)
np.random.seed(42)
N_SENSORS = 6
sensors = [f'S{i+1}' for i in range(N_SENSORS)]

start = pd.Timestamp('2025-11-01 00:00')
periods = 24 * 6 # six measurements per hour for 24 hours
freq = '10min' # 10-minute frequency
idx = pd.date_range(start=start, periods=periods, freq=freq)

rows = []
for sensor in sensors:
    base_temp = 15 + 5 * np.random.rand() # baseline temp different per sensor
    base_hum = 40 + 20 * np.random.rand()
    noise_t = np.random.normal(0, 1.5, size=len(idx))
    noise_h = np.random.normal(0, 5.0, size=len(idx))
    # add a diurnal temperature variation
    hours = (idx.hour + idx.minute/60.0)
    diurnal = 6 * np.sin(2 * np.pi * (hours - 6) / 24)
    
    temps = base_temp + diurnal + noise_t
    hums = np.clip(base_hum - 0.5 * diurnal + noise_h, 0, 100)
    
    for ts, t, h in zip(idx, temps, hums):
        rows.append({'sensor_id': sensor, 'timestamp': ts, 'temperature': t, 'humidity': h})

sensor_df = pd.DataFrame(rows)

# add some missing values intentionally and some outliers
sensor_df.loc[sensor_df.sample(frac=0.01, random_state=1).index, 'temperature'] = np.nan
sensor_df.loc[sensor_df.sample(frac=0.005, random_state=2).index, 'humidity'] = np.nan
# inject outliers
outlier_idx = sensor_df.sample(frac=0.002, random_state=3).index
sensor_df.loc[outlier_idx, 'temperature'] += np.random.choice([20, -15], size=len(outlier_idx))

print('--- sensor_df (head) ---')
print(sensor_df.head())

# sensor metadata (wide)
meta = pd.DataFrame({
'sensor_id': sensors,
'location': ['Room A', 'Room B', 'Room C', 'Room A', 'Room D', 'Room B'],
'type': ['thermo-hygro', 'thermo-hygro', 'thermo', 'thermo-hygro', 'hygro', 'thermo']
})

# events table (sparse)
events = pd.DataFrame([
{'sensor_id': 'S1', 'timestamp': start + pd.Timedelta(hours=5), 'event': 'maintenance'},
{'sensor_id': 'S3', 'timestamp': start + pd.Timedelta(hours=8, minutes=20), 'event': 'calibration'},
{'sensor_id': 'S2', 'timestamp': start + pd.Timedelta(hours=18), 'event': 'spike_detected'}
])

--- sensor_df (head) ---
  sensor_id           timestamp  temperature   humidity
0        S1 2025-11-01 00:00:00    11.844233  55.829533
1        S1 2025-11-01 00:10:00    13.162956  55.409148
2        S1 2025-11-01 00:20:00    10.544302  64.612578
3        S1 2025-11-01 00:30:00          NaN  63.473544
4        S1 2025-11-01 00:40:00    13.332673  63.221174


In [69]:
# -------------------------------
# 2) MultiIndex: set index as (sensor_id, timestamp)
# -------------------------------
sensor_mi = sensor_df.set_index(['sensor_id', 'timestamp']).sort_index()
print('\n--- sensor_mi (index sample) ---')
print(sensor_mi.head())

# show selecting using MultiIndex
print('\nSelect all readings for S2 between two times:')
print(sensor_mi.loc['S2'].between_time('06:00', '09:00').head())


--- sensor_mi (index sample) ---
                               temperature   humidity
sensor_id timestamp                                  
S1        2025-11-01 00:00:00    11.844233  55.829533
          2025-11-01 00:10:00    13.162956  55.409148
          2025-11-01 00:20:00    10.544302  64.612578
          2025-11-01 00:30:00          NaN  63.473544
          2025-11-01 00:40:00    13.332673  63.221174

Select all readings for S2 between two times:
                     temperature   humidity
timestamp                                  
2025-11-01 06:00:00    16.292123  38.803557
2025-11-01 06:10:00    16.560178  53.083026
2025-11-01 06:20:00    15.860508  54.342884
2025-11-01 06:30:00    14.793219  44.742754
2025-11-01 06:40:00    16.511539  48.741353


In [70]:
# -------------------------------
# 3) Multi-level columns example: create aggregated summary with MultiIndex columns
# -------------------------------
agg_funcs = {
'temperature': ['mean', 'std', 'min', 'max'],
'humidity': ['mean', 'std']
}
summary_by_sensor = sensor_df.groupby('sensor_id').agg(agg_funcs)
# flatten/unflatten demonstration: keep as MultiIndex columns
print('\n--- summary_by_sensor (MultiIndex columns) ---')
print(summary_by_sensor.head())

# rename columns to readable MultiIndex (level names)
summary_by_sensor.columns.names = ['measurement', 'stat']
print('\nColumns levels:', summary_by_sensor.columns.names)

# access a specific subtable using xs
print('\nTemperature means:')
print(summary_by_sensor.xs('temperature', axis=1)[['mean', 'std']])


--- summary_by_sensor (MultiIndex columns) ---
          temperature                                  humidity          
                 mean       std       min        max       mean       std
sensor_id                                                                
S1          16.718369  4.511676  8.552629  25.213955  59.259889  5.560005
S2          15.427998  4.497470  2.122613  24.023564  45.599224  5.493384
S3          15.658394  4.421943  7.273332  25.556400  56.385015  5.146519
S4          15.668049  4.238744  7.365955  24.204889  48.812680  5.493305
S5          17.801163  4.889507  9.133021  38.594542  49.913718  5.367881

Columns levels: ['measurement', 'stat']

Temperature means:
stat            mean       std
sensor_id                     
S1         16.718369  4.511676
S2         15.427998  4.497470
S3         15.658394  4.421943
S4         15.668049  4.238744
S5         17.801163  4.889507
S6         18.627710  4.633420


In [71]:
summary_by_sensor.columns = [
    f"{measurement}_{stat}"
    for measurement, stat in summary_by_sensor.columns
]

print('\n--- summary_by_sensor (flat columns) ---')
print(summary_by_sensor.head())


--- summary_by_sensor (flat columns) ---
           temperature_mean  temperature_std  temperature_min  \
sensor_id                                                       
S1                16.718369         4.511676         8.552629   
S2                15.427998         4.497470         2.122613   
S3                15.658394         4.421943         7.273332   
S4                15.668049         4.238744         7.365955   
S5                17.801163         4.889507         9.133021   

           temperature_max  humidity_mean  humidity_std  
sensor_id                                                
S1               25.213955      59.259889      5.560005  
S2               24.023564      45.599224      5.493384  
S3               25.556400      56.385015      5.146519  
S4               24.204889      48.812680      5.493305  
S5               38.594542      49.913718      5.367881  


In [72]:
# -------------------------------
# 4) Advanced groupby with named aggregation and custom functions
# -------------------------------
# We'll compute per-sensor and per-location aggregations using named aggregations
sensor_with_meta = sensor_df.merge(meta, on='sensor_id', how='left')

agg_named = sensor_with_meta.groupby(['location']).agg(
temp_mean=('temperature', 'mean'),
temp_med=('temperature', 'median'),
temp_iqr=('temperature', lambda x: np.subtract(*np.percentile(x.dropna(), [75, 25]))),
humidity_mean=('humidity', 'mean'),
n_readings=('temperature', 'count')
)
print('\n--- agg_named by location ---')
print(agg_named)

# custom aggregation using apply (slower but expressive) -> compute time-of-day sensitivity

def day_night_diff(group):
    group = group.copy()
    # daytime mean vs nighttime mean
    group['hour'] = group['timestamp'].dt.hour
    daytime = group.loc[(group['hour'] >= 6) & (group['hour'] < 18), 'temperature']
    nighttime = group.loc[(group['hour'] < 6) | (group['hour'] >= 18), 'temperature']
    return pd.Series({'day_minus_night': daytime.mean() - nighttime.mean()})

sens_daynight = sensor_df.merge(meta, on='sensor_id').groupby('sensor_id').apply(day_night_diff, include_groups=False)
print('\n--- day_minus_night per sensor ---')
print(sens_daynight.head())


--- agg_named by location ---
          temp_mean   temp_med  temp_iqr  humidity_mean  n_readings
location                                                           
Room A    16.187642  16.260233  7.800304      54.036285         283
Room B    17.022280  16.570175  7.810141      48.698361         287
Room C    15.658394  15.917286  8.167460      56.385015         143
Room D    17.801163  18.022922  8.284296      49.913718         142

--- day_minus_night per sensor ---
           day_minus_night
sensor_id                 
S1                7.691988
S2                7.222448
S3                7.461552
S4                7.165945
S5                7.980190


In [73]:
# Note on merge vs join
# pandas.merge() is the core function for all merges/joins.
# df.merge() and df.join() are convenient wrappers:
# - df.join() defaults to left join on df1's index and df2's index.
# - df.merge() defaults to inner join on columns, but can join on indexes.
# Use merge() for flexibility; join() can save typing in simple index-based left joins.

In [74]:
# -------------------------------
# 5) pivot_table & crosstab
# -------------------------------
# pivot_table: average temperature per sensor per hour of day
sensor_df['hour'] = sensor_df['timestamp'].dt.hour
pv = pd.pivot_table(sensor_df, values='temperature', index='sensor_id', columns='hour', aggfunc='mean')
print('\n--- pivot_table mean temperature by sensor x hour ---')
print(pv.iloc[:, :6]) # show first 6 hours as sample

# crosstab: count of readings per sensor per type
sensor_meta_joined = sensor_df.merge(meta, on='sensor_id')
ct = pd.crosstab(sensor_meta_joined['sensor_id'], sensor_meta_joined['type'])
print('\n--- crosstab counts sensor x type ---')
print(ct)


--- pivot_table mean temperature by sensor x hour ---
hour               0          1          2          3          4          5
sensor_id                                                                  
S1         12.210048  10.664322  10.720927  12.989877  14.358660  14.856285
S2          9.690962  10.132154  10.812555  12.572447  13.260633  14.696669
S3         10.254658  10.215027  10.674089  13.038813  12.860999  15.427306
S4          9.326482  10.789089  11.722838  13.088461  13.111283  14.757508
S5         12.241501  12.651350  11.540636  13.750657  14.472117  16.459888
S6         13.411637  12.972633  14.169101  15.001611  16.085027  17.580157

--- crosstab counts sensor x type ---
type       hygro  thermo  thermo-hygro
sensor_id                             
S1             0       0           144
S2             0       0           144
S3             0     144             0
S4             0       0           144
S5           144       0             0
S6             0     144 

In [75]:
# -------------------------------
# 6) Merging and joining examples (all possibilities)
# -------------------------------
# inner merge (only sensors present in both)
m_inner = sensor_df.merge(meta, on='sensor_id', how='inner')
# left merge (keeps all readings)
m_left = sensor_df.merge(meta, on='sensor_id', how='left')
# right and outer
m_right = sensor_df.merge(meta, on='sensor_id', how='right')
m_outer = sensor_df.merge(meta, on='sensor_id', how='outer')

print('\nmerge sizes: inner={}, left={}, right={}, outer={}'.format(
len(m_inner), len(m_left), len(m_right), len(m_outer)
))

# join on index: prepare two DataFrames with indexes
df_a = sensor_df.sample(50, random_state=4).set_index(['sensor_id', 'timestamp']).sort_index()
df_b = pd.DataFrame({'battery': np.random.randint(20, 100, size=30)},
index=df_a.index[:30])
# left join using index
joined = df_a.join(df_b, how='left')

print('\n--- joined (index join) sample ---')
print(joined.head())

# concat: stack vertically and horizontally
concat_v = pd.concat([sensor_df.head(5), sensor_df.tail(5)], axis=0)
concat_h = pd.concat([sensor_df.head(5).reset_index(drop=True).iloc[:, :3],
sensor_df.head(5).reset_index(drop=True).iloc[:, 3:]], axis=1)

# combine_first: combine two sources preferring left's non-null values
left = sensor_df.head(10).copy()
right = left.copy()
right.loc[right.sample(frac=0.3, random_state=5).index, 'temperature'] = np.nan
combined = right.combine_first(left)
print('\n--- combine_first example ---')
print(combined.head())


merge sizes: inner=864, left=864, right=864, outer=864

--- joined (index join) sample ---
                               temperature   humidity  hour  battery
sensor_id timestamp                                                 
S1        2025-11-01 07:50:00    16.998632  51.400346     7     74.0
          2025-11-01 10:30:00    24.450338  75.506305    10     35.0
          2025-11-01 11:00:00    23.210710  60.886517    11     81.0
          2025-11-01 11:10:00    21.762797  59.342354    11     67.0
          2025-11-01 11:40:00    22.796130  52.161576    11     49.0

--- combine_first example ---
  sensor_id           timestamp  temperature   humidity  hour
0        S1 2025-11-01 00:00:00    11.844233  55.829533     0
1        S1 2025-11-01 00:10:00    13.162956  55.409148     0
2        S1 2025-11-01 00:20:00    10.544302  64.612578     0
3        S1 2025-11-01 00:30:00          NaN  63.473544     0
4        S1 2025-11-01 00:40:00    13.332673  63.221174     0


In [76]:
# -------------------------------
# 7) Reshaping: stack/unstack and melt
# -------------------------------
# create a small pivot and show stack/unstack
small = sensor_df[sensor_df['sensor_id'].isin(['S1', 'S2'])].head(12)
pv_small = small.pivot(index='timestamp', columns='sensor_id', values='temperature')
print('\n--- pv_small pivot (timestamp x sensor) ---')
print(pv_small.head())
print('\nstacked -> unstacked roundtrip (demonstration)')
stacked = pv_small.stack()
print(stacked.head())
print(stacked.unstack().head())

# melt: convert wide back to long
melted = pv_small.reset_index().melt(id_vars='timestamp', var_name='sensor_id', value_name='temperature')
print('\n--- melted back to long ---')
print(melted.head())


--- pv_small pivot (timestamp x sensor) ---
sensor_id                   S1
timestamp                     
2025-11-01 00:00:00  11.844233
2025-11-01 00:10:00  13.162956
2025-11-01 00:20:00  10.544302
2025-11-01 00:30:00        NaN
2025-11-01 00:40:00  13.332673

stacked -> unstacked roundtrip (demonstration)
timestamp            sensor_id
2025-11-01 00:00:00  S1           11.844233
2025-11-01 00:10:00  S1           13.162956
2025-11-01 00:20:00  S1           10.544302
2025-11-01 00:30:00  S1                 NaN
2025-11-01 00:40:00  S1           13.332673
dtype: float64
sensor_id                   S1
timestamp                     
2025-11-01 00:00:00  11.844233
2025-11-01 00:10:00  13.162956
2025-11-01 00:20:00  10.544302
2025-11-01 00:30:00        NaN
2025-11-01 00:40:00  13.332673

--- melted back to long ---
            timestamp sensor_id  temperature
0 2025-11-01 00:00:00        S1    11.844233
1 2025-11-01 00:10:00        S1    13.162956
2 2025-11-01 00:20:00        S1    10.54430

In [77]:
# -------------------------------
# 8) Query: expressive filtering
# -------------------------------
# Use DataFrame.query for readable boolean expressions
q = sensor_df.query('temperature > 25 and humidity < 50')
print(f'\nNumber of readings with temperature>25 and humidity<50: {len(q)}')

# boolean indexing vs query for more complex expressions
expr = "(temperature > temperature.mean()) & (humidity < humidity.quantile(0.25))"
# careful: can't use temperature.mean() inside query easily; compute first
temp_mean = sensor_df['temperature'].mean()
hum_q25 = sensor_df['humidity'].quantile(0.25)
q2 = sensor_df.query('@temp_mean < temperature and humidity < @hum_q25')
print(f'\nUsing external variables in query, matches: {len(q2)}')


Number of readings with temperature>25 and humidity<50: 12

Using external variables in query, matches: 139


In [78]:
# -------------------------------
# 9) Aggregations with groupby + transform and filter
# -------------------------------
# find sensors whose median temperature is > global median
global_median = sensor_df['temperature'].median()
sensor_medians = sensor_df.groupby('sensor_id')['temperature'].median()
hot_sensors = sensor_medians[sensor_medians > global_median].index.tolist()
print('\nSensors with median temperature > global median:', hot_sensors)

# use transform to broadcast group statistic back to rows
sensor_df['sensor_temp_median'] = sensor_df.groupby('sensor_id')['temperature'].transform('median')
# filter rows where temperature is above group's median
above_group_median = sensor_df[sensor_df['temperature'] > sensor_df['sensor_temp_median']]
print('\nRows above their sensor median (sample):')
print(above_group_median.head())

# groupby.filter to keep only sensors with at least 200 readings
sensors_with_enough = sensor_df.groupby('sensor_id').filter(lambda g: len(g) >= 200)
print('\nSensors kept (len>=200):', sensors_with_enough['sensor_id'].unique())


Sensors with median temperature > global median: ['S1', 'S5', 'S6']

Rows above their sensor median (sample):
   sensor_id           timestamp  temperature   humidity  hour  \
29        S1 2025-11-01 04:50:00    17.846883  64.052320     4   
32        S1 2025-11-01 05:20:00    17.064629  58.211946     5   
37        S1 2025-11-01 06:10:00    17.429709  61.295790     6   
38        S1 2025-11-01 06:20:00    18.503335  57.635505     6   
39        S1 2025-11-01 06:30:00    17.912910  62.192710     6   

    sensor_temp_median  
29            17.01069  
32            17.01069  
37            17.01069  
38            17.01069  
39            17.01069  

Sensors kept (len>=200): <StringArray>
[]
Length: 0, dtype: str


In [79]:
# -------------------------------
# 10) MultiIndex columns: produce aggregated table with hierarchical columns and plot
# -------------------------------
agg_multi = sensor_df.groupby('sensor_id').agg(
temp_mean=('temperature', 'mean'),
temp_std=('temperature', 'std'),
hum_mean=('humidity', 'mean'),
hum_std=('humidity', 'std')
)
# create MultiIndex columns manually
agg_multi.columns = pd.MultiIndex.from_tuples([('temperature', 'mean'), ('temperature', 'std'), ('humidity', 'mean'), ('humidity', 'std')])
print('\n--- agg_multi (MultiIndex columns) ---')
print(agg_multi)


--- agg_multi (MultiIndex columns) ---
          temperature             humidity          
                 mean       std       mean       std
sensor_id                                           
S1          16.718369  4.511676  59.259889  5.560005
S2          15.427998  4.497470  45.599224  5.493384
S3          15.658394  4.421943  56.385015  5.146519
S4          15.668049  4.238744  48.812680  5.493305
S5          17.801163  4.889507  49.913718  5.367881
S6          18.627710  4.633420  51.732933  5.117629


In [80]:
# -------------------------------
# 11) Rolling, expanding and time-based resampling (for sensor S2)
# -------------------------------
s2 = sensor_df[sensor_df['sensor_id'] == 'S2'].set_index('timestamp').sort_index()
# rolling 30-minute window (freq is 10min so 3 observations -> use time-based window)
s2['temp_roll_30min'] = s2['temperature'].rolling('30min').mean()
# humidity diff next reading -> shift(-1)
s2['humidity_next_diff'] = s2['humidity'].shift(-1) - s2['humidity']
# boolean: temp increased >5C and humidity dropped >20 compared to previous measurement
s2['temp_prev_diff'] = s2['temperature'] - s2['temperature'].shift(1)
s2['hum_prev_diff'] = s2['humidity'].shift(1) - s2['humidity']
s2['extreme_change'] = (s2['temp_prev_diff'] > 5) & (s2['hum_prev_diff'] > 20)

    ## Task Specification: Aggregations, Joins, GroupBy, and Rolling Analysis with Pandas

---

### **Task Overview**
This assignment simulates a **customer purchase analytics workflow** using Pandas. You will work with timestamped transaction data and customer metadata, performing:

- Group-based aggregations
- Time-based analysis
- Rolling window calculations within groups
- Joining customer metadata from another dataset

---

### **Datasets**
You are provided with two datasets:

| File | Description |
|------|-------------|
| `data/purchases.csv` | Timestamped purchase data (contains `Customer_id`, `Timestamp`, `Price`, etc.) |
| `data/customer_info.csv` | Customer metadata (contains `Customer_id`, `Born_date`, `City`, etc.) |

---

### **1. Load and Prepare the Data**
- **Task 1.1:** Load the two CSV files into Pandas DataFrames:
  - Use `df` for `purchases.csv`
  - Use `customer_info` for `customer_info.csv`
  - Shuffle the rows of `df` right after loading: `df = df.sample(frac=1, random_state=42)`

- **Task 1.2:** Convert the `Timestamp` column in `df` to Pandas `datetime` format.

- **Task 1.3:** Set `Timestamp` as the **index** of `df` and **sort** the index in ascending order.

---

### **2. Join Customer Metadata**
- **Task 2.1:** Merge the purchase DataFrame (`df`) with the customer metadata DataFrame (`customer_info`) on the `Customer_id` column.

- **Task 2.2:** The resulting DataFrame should include the columns `Born_date` and `City` from the `customer_info` DataFrame.

---

### **3. GroupBy and Aggregations**
Using the merged DataFrame:

- **Task 3.1:** Group by **Customer_id** and compute:
  - Total revenue (`sum(Price)`)
  - Number of purchases (`count`)
  - Average purchase price (`mean(Price)`)

- **Task 3.2:** Create a new feature by extracting the **month name** (e.g., `"January"`, `"February"`) from the `Timestamp`.

- **Task 3.3:** Group by **City** and **Month** and compute:
  - Total monthly revenue per city (`sum(Price)`)

---

### **4. Rolling Window Analysis (Per Customer)**
Perform rolling window calculations **within each customer group**:

- Inside each customer group:
  - **Sort** the DataFrame by `Timestamp`.

- Compute the following:
  - A **3-purchase rolling average** of `Price`.
  - A `diff()` column showing the change in price compared to the previous purchase.

> **Note:** The rolling operations must be computed **within each customer group**, not across all customers.

---

### **5. Advanced Filtering**
- Do the same as in 4 only for customers from **Berlin**, make sure only necessary values are joined (that is filter first, join second).

- Do the same as in 4 only for rows where the `Price` of the current purchase is greater than **20%** than the previous purchase (`Price`), make sure only necessary values are joined (that is filter first, join second).

---

### **6. Custom Lambda Function for GroupBy Operations**
- Apply a **custom lambda function** within a `groupby()` operation to filter or modify the data. For example:
  - Use a lambda to **filter** customers who have purchased more than a specific amount within a given timeframe (e.g., greater than $1000 in total purchases during the last quarter).
  - Apply custom transformations to grouped data using the lambda function.

---

### 7. Time-Based Analysis
- Perform additional time-based analysis:
  - Calculate **monthly growth rate** for total revenue per customer (i.e., compare total revenue of the current month with the previous month): `growth_rate = (revenue_current_month - revenue_previous_month) / revenue_previous_month` (`NaN` for the first month).
  - Create a **cumulative sum** of purchases over time per customer.


### 1. Load and Prepare the Data

I loaded `purchases.csv` into the DataFrame `df` and `customer_info.csv` into `customer_info`. Right after loading, I shuffled the rows of `df` with `df.sample(frac=1, random_state=42)` so that the following steps do not depend on the original row order.

Next, I converted the `Timestamp` column to the `datetime` type, set it as the index of `df` and sorted the index in ascending order. This makes the data chronologically ordered and enables time-based operations such as `index.month_name()` or `pd.Grouper`.

In [81]:
df = pd.read_csv("data/purchases.csv")
df = df.sample(frac=1, random_state=42)  # shuffle right after loading
customer_info = pd.read_csv("data/customer_info.csv")

df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.set_index("Timestamp").sort_index()
df.head(10)

,Customer_id,Price
Timestamp,,
2025-01-01 00:00:00,C037,436.71
2025-01-01 00:30:00,C049,274.70
2025-01-01 01:00:00,C017,380.42
2025-01-01 01:30:00,C049,378.28
2025-01-01 02:00:00,C002,13.88
2025-01-01 02:30:00,C002,67.50
2025-01-01 03:00:00,C028,331.52
2025-01-01 03:30:00,C023,85.00
2025-01-01 04:00:00,C037,131.47


### 2. Join Customer Metadata

I joined the purchases with the customer metadata on `Customer_id` using `df.join(..., on="Customer_id", how="left")`. I used `join` instead of `merge`, because `merge` discards the index and the `Timestamp` index would be lost.

From `customer_info` I kept only `Customer_id`, `Born_date` and `City`. The resulting DataFrame `customer` therefore contains the original purchase columns plus `Born_date` and `City`. I also checked that `Customer_id` is unique in `customer_info`, so the join does not duplicate any purchases.

In [82]:
customer = df.join(customer_info[["Customer_id", "Born_date", "City"]].set_index("Customer_id"),on="Customer_id",how="left",)

### 3. GroupBy and Aggregations

**3.1** I grouped the merged data by `Customer_id` and computed the total revenue (`sum`), the number of purchases (`count`) and the average purchase price (`mean`) of `Price`.

**3.2** I created the column `month` with the month name (e.g. "January") extracted from the `Timestamp` index using `index.month_name()`. I also added the helper column `month_num`.

**3.3** I grouped the data by `City` and `month` and computed the total monthly revenue per city. 

In [83]:
customer.groupby(["Customer_id"]).agg(
    Total_revenue=('Price', 'sum'),
    Number_of_purchases=('Price', 'count'),
    Average_purchase_value=('Price', 'mean')
)

,Total_revenue,Number_of_purchases,Average_purchase_value
Customer_id,,,
C001,13055.75,56,233.138393
C002,15927.80,64,248.871875
C003,14947.34,56,266.916786
C004,15111.43,59,256.125932
C005,16798.61,64,262.478281
C006,14774.24,63,234.511746
C007,11886.07,48,247.626458
C008,15555.66,65,239.317846
C009,12630.37,52,242.891731


In [84]:
customer["month"] = customer.index.month_name()
customer.groupby(["City", "month"]).agg(Price_City_Month=("Price", "sum"))

Price_City_Month
City      month                     
Berlin    February          86163.85
          January          101192.17
          March             10474.86
Cologne   February          86781.31
          January           98030.00
          March             11220.64
Frankfurt February          52020.01
          January           61492.29
          March              6698.88
Hamburg   February          42056.17
          January           43988.43
          March              5340.12
Munich    February          67799.84
          January           78726.19
          March              7330.35

### 4. Rolling Window Analysis (Per Customer)

I sorted `customer` by the `Timestamp` index and grouped it by `Customer_id`. Inside each group I computed:

- `price_roll_3`: a rolling average of `Price` over 3 purchases, using `transform` with a lambda function,
- `change_of_price`: the difference of `Price` compared to the previous purchase of the same customer, using `diff()`.

Because both columns are computed within each customer group, the values of different customers are never mixed. The first two purchases of every customer have `NaN` in `price_roll_3` (a window of 3 values is required) and the first purchase has `NaN` in `change_of_price` (there is no previous purchase).

In [85]:
customer = customer.sort_index()
grouped = customer.groupby("Customer_id")["Price"]
customer["price_roll_3"] = grouped.transform(lambda s: s.rolling(3).mean())
customer["change_of_price"] = grouped.diff()
customer.head(10)

,Customer_id,Price,Born_date,City,month,price_roll_3,change_of_price
Timestamp,,,,,,,
2025-01-01 00:00:00,C037,436.71,1996-09-14,Munich,January,NaN,NaN
2025-01-01 00:30:00,C049,274.70,1979-01-20,Munich,January,NaN,NaN
2025-01-01 01:00:00,C017,380.42,1981-02-23,Cologne,January,NaN,NaN
2025-01-01 01:30:00,C049,378.28,1979-01-20,Munich,January,NaN,103.58
2025-01-01 02:00:00,C002,13.88,1988-07-03,Frankfurt,January,NaN,NaN
2025-01-01 02:30:00,C002,67.50,1988-07-03,Frankfurt,January,NaN,53.62
2025-01-01 03:00:00,C028,331.52,2003-05-03,Berlin,January,NaN,NaN
2025-01-01 03:30:00,C023,85.00,1961-08-08,Berlin,January,NaN,NaN
2025-01-01 04:00:00,C037,131.47,1996-09-14,Munich,January,NaN,-305.24


### 5. Advanced Filtering

**Berlin customers.** I applied the principle *filter first, join second*. First, I selected only the Berlin customers from `customer_info`. Then I joined them with the purchases using an inner join, so only the purchases of Berlin customers remain. Finally, I computed `price_roll_3` and `change_of_price` in a new `groupby` on the filtered data, as in section 4.

**Purchases with price more than 20 % higher than the previous one.** I computed `prev_price` (the previous price of the same customer) on the purchases only, and kept the rows where `Price > 1.2 * prev_price`. Only after that I joined the customer metadata. The first purchase of each customer has `prev_price = NaN`, so it is filtered out automatically.

I interpreted the task literally ("do the same as in 4 only for these rows"), so the rolling average and `diff()` are computed on the filtered rows. Therefore many values are `NaN`: after filtering, each customer has only a few rows left, `rolling(3)` needs at least 3 rows and `diff()` needs a previous row of the same customer. Also, `change_of_price` here is the difference between two *filtered* purchases, which are not necessarily consecutive in the original data.

In [86]:
berlin_info = customer_info.loc[customer_info["City"] == "Berlin", ["Customer_id", "Born_date", "City"]].set_index("Customer_id")
customer_Berlin = df.join(berlin_info, on="Customer_id", how="inner").sort_index()

g = customer_Berlin.groupby("Customer_id")["Price"]
customer_Berlin["price_roll_3"] = g.transform(lambda s: s.rolling(3).mean())
customer_Berlin["change_of_price"] = g.diff()
customer_Berlin.head(10)

,Customer_id,Price,Born_date,City,price_roll_3,change_of_price
Timestamp,,,,,,
2025-01-01 03:00:00,C028,331.52,2003-05-03,Berlin,NaN,NaN
2025-01-01 03:30:00,C023,85.00,1961-08-08,Berlin,NaN,NaN
2025-01-01 07:00:00,C044,51.61,1984-12-10,Berlin,NaN,NaN
2025-01-01 07:30:00,C026,175.11,1971-09-03,Berlin,NaN,NaN
2025-01-01 11:30:00,C024,223.33,1980-07-22,Berlin,NaN,NaN
2025-01-01 12:30:00,C034,97.43,1974-12-05,Berlin,NaN,NaN
2025-01-01 19:00:00,C036,308.67,1962-12-23,Berlin,NaN,NaN
2025-01-01 20:00:00,C026,449.11,1971-09-03,Berlin,NaN,274.00
2025-01-01 23:00:00,C041,367.65,1998-09-15,Berlin,NaN,NaN


In [87]:
df_prev = df.copy()
df_prev["prev_price"] = df_prev.groupby("Customer_id")["Price"].shift(1)
df_20 = df_prev[df_prev["Price"] > 1.2 * df_prev["prev_price"]]

customer_20 = df_20.join(customer_info[["Customer_id", "Born_date", "City"]].set_index("Customer_id"),on="Customer_id",how="left",)


g20 = customer_20.groupby("Customer_id")["Price"]
customer_20["price_roll_3"] = g20.transform(lambda s: s.rolling(3).mean())
customer_20["change_of_price"] = g20.diff()
customer_20.head(10)

,Customer_id,Price,prev_price,Born_date,City,price_roll_3,change_of_price
Timestamp,,,,,,,
2025-01-01 01:30:00,C049,378.28,274.70,1979-01-20,Munich,NaN,NaN
2025-01-01 02:30:00,C002,67.50,13.88,1988-07-03,Frankfurt,NaN,NaN
2025-01-01 06:30:00,C002,278.49,67.50,1988-07-03,Frankfurt,NaN,210.99
2025-01-01 09:00:00,C032,400.18,144.80,1987-10-03,Munich,NaN,NaN
2025-01-01 14:00:00,C011,379.95,228.59,1970-02-09,Hamburg,NaN,NaN
2025-01-01 16:30:00,C006,448.99,51.55,1980-01-10,Frankfurt,NaN,NaN
2025-01-01 17:00:00,C016,262.49,159.61,1962-06-16,Cologne,NaN,NaN
2025-01-01 20:00:00,C026,449.11,175.11,1971-09-03,Berlin,NaN,NaN
2025-01-01 21:00:00,C019,255.99,184.73,1983-12-24,Cologne,NaN,NaN


### 6. Custom Lambda Function for GroupBy Operations

**6.1 Filtering with a lambda.** I selected the purchases from the last quarter of the dataset (the last 3 months counted from the latest timestamp in the data, not from today's date). Then I used `groupby("Customer_id").filter(lambda g: g["Price"].sum() > 1000)`, which keeps only the customers whose total purchases in this period exceed 1000. The number of such customers is [doplň].

**6.2 Transformation with a lambda.** I used `groupby(...).transform(lambda ...)` to create two new columns:

- `price_zscore`: the price standardized within each customer, `(price - mean) / std`,
- `price_share`: the share of each purchase in the customer's total revenue.

Customers with a single purchase have `NaN` in `price_zscore`, because the standard deviation of one value is undefined.

In [88]:
last_ts = customer.index.max()
quarter_start = last_ts - pd.DateOffset(months=3)  

last_quarter = customer.loc[customer.index > quarter_start]
big_spenders = last_quarter.groupby("Customer_id").filter(lambda g: g["Price"].sum() > 1000)

print("Customers with > 1000 in the last quarter:",big_spenders["Customer_id"].nunique())
big_spenders.head(10)

Customers with > 1000 in the last quarter: 50


,Customer_id,Price,Born_date,City,month,price_roll_3,change_of_price
Timestamp,,,,,,,
2025-01-01 00:00:00,C037,436.71,1996-09-14,Munich,January,NaN,NaN
2025-01-01 00:30:00,C049,274.70,1979-01-20,Munich,January,NaN,NaN
2025-01-01 01:00:00,C017,380.42,1981-02-23,Cologne,January,NaN,NaN
2025-01-01 01:30:00,C049,378.28,1979-01-20,Munich,January,NaN,103.58
2025-01-01 02:00:00,C002,13.88,1988-07-03,Frankfurt,January,NaN,NaN
2025-01-01 02:30:00,C002,67.50,1988-07-03,Frankfurt,January,NaN,53.62
2025-01-01 03:00:00,C028,331.52,2003-05-03,Berlin,January,NaN,NaN
2025-01-01 03:30:00,C023,85.00,1961-08-08,Berlin,January,NaN,NaN
2025-01-01 04:00:00,C037,131.47,1996-09-14,Munich,January,NaN,-305.24


In [89]:
customer["price_zscore"] = customer.groupby("Customer_id")["Price"].transform(lambda s: (s - s.mean()) / s.std())
customer["price_share"] = customer.groupby("Customer_id")["Price"].transform(lambda s: s / s.sum())

customer[["Customer_id", "Price", "price_zscore", "price_share"]].head(10)

,Customer_id,Price,price_zscore,price_share
Timestamp,,,,
2025-01-01 00:00:00,C037,436.71,1.169893,0.023330
2025-01-01 00:30:00,C049,274.70,0.216955,0.015159
2025-01-01 01:00:00,C017,380.42,0.999162,0.022854
2025-01-01 01:30:00,C049,378.28,0.970418,0.020876
2025-01-01 02:00:00,C002,13.88,-1.684600,0.000871
2025-01-01 02:30:00,C002,67.50,-1.300211,0.004238
2025-01-01 03:00:00,C028,331.52,0.580172,0.022268
2025-01-01 03:30:00,C023,85.00,-1.386779,0.004976
2025-01-01 04:00:00,C037,131.47,-0.893484,0.007023


### 7. Time-Based Analysis

**7.1 Monthly growth rate.** I summed the revenue per customer and calendar month using `pd.Grouper(freq="MS")` and then computed the growth rate within each customer with `pct_change()`, which is exactly `(revenue_current_month - revenue_previous_month) / revenue_previous_month`. The first month of each customer is `NaN`. Months in which a customer made no purchase do not appear in the table, so the growth rate is computed against the last month with a purchase.

**7.2 Cumulative sum.** Since `customer` is sorted by time, `groupby("Customer_id")["Price"].cumsum()` gives the cumulative revenue of each customer over time in the new column `cumulative_revenue`.

In [90]:

monthly = (customer.groupby(["Customer_id", pd.Grouper(freq="MS")])["Price"]
    .sum()
    .rename("monthly_revenue")
    .reset_index())

monthly["growth_rate"] = monthly.groupby("Customer_id")["monthly_revenue"].pct_change()
monthly.head(10)

,Customer_id,Timestamp,monthly_revenue,growth_rate
0,C001,2025-01-01,7302.10,NaN
1,C001,2025-02-01,5204.79,-0.287220
2,C001,2025-03-01,548.86,-0.894547
3,C002,2025-01-01,7538.37,NaN
4,C002,2025-02-01,7838.50,0.039814
5,C002,2025-03-01,550.93,-0.929715
6,C003,2025-01-01,7133.09,NaN
7,C003,2025-02-01,7141.33,0.001155
8,C003,2025-03-01,672.92,-0.905771
9,C004,2025-01-01,6920.10,NaN


In [91]:
customer["cumulative_revenue"] = customer.groupby("Customer_id")["Price"].cumsum()

customer[["Customer_id", "Price", "cumulative_revenue"]].head(10)

,Customer_id,Price,cumulative_revenue
Timestamp,,,
2025-01-01 00:00:00,C037,436.71,436.71
2025-01-01 00:30:00,C049,274.70,274.70
2025-01-01 01:00:00,C017,380.42,380.42
2025-01-01 01:30:00,C049,378.28,652.98
2025-01-01 02:00:00,C002,13.88,13.88
2025-01-01 02:30:00,C002,67.50,81.38
2025-01-01 03:00:00,C028,331.52,331.52
2025-01-01 03:30:00,C023,85.00,85.00
2025-01-01 04:00:00,C037,131.47,568.18
